In [9]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [10]:
# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# === ChatPromptTemplate에 MessagesPlaceholder 배치 ===
# system → 대화 히스토리(MessagesPlaceholder) → 현재 질문 순서로 구성합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 답하세요.\n\n[document]\n{document}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

print(f"필요한 변수: {prompt.input_variables}")

필요한 변수: ['document', 'history', 'question']


In [12]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)
parser = StrOutputParser()

# === 체인 구성 ===
chain = prompt | model | parser

# === 첫 번째 질문: 히스토리 없음 ===
result1 = chain.invoke({
    "document": document,
    "history": [],  # 첫 대화이므로 빈 리스트
    "question": "어댑터즈는 무엇인가요?"
})
print(f"Q: 어댑터즈는 무엇인가요?")
print(f"A: {result1}")

Q: 어댑터즈는 무엇인가요?
A: 어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.


In [13]:
# === 두 번째 질문: 이전 대화를 히스토리에 포함 ===
history = [
    HumanMessage(content="어댑터즈는 무엇인가요?"),
    AIMessage(content="어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다."),
]

result2 = chain.invoke({
    "document": document,
    "history": history,
    "question": "거기서 사용하는 교수법은 뭔가요?"
})
print(f"Q: 거기서 사용하는 교수법은 뭔가요?")
print(f"A: {result2}")

Q: 거기서 사용하는 교수법은 뭔가요?
A: 거기서 사용하는 교수법은 '5단 분석법'입니다.


In [14]:
# === 세 번째 질문: 히스토리 누적 ===
history = [
    HumanMessage(content="어댑터즈는 무엇인가요?"),
    AIMessage(content="어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다."),
    HumanMessage(content="거기서 사용하는 교수법은 뭔가요?"),
    AIMessage(content="어댑터즈에서 사용하는 교수법은 5단 분석법입니다."),
]

result3 = chain.invoke({
    "document": document,
    "history": history,
    "question": "그 교수법의 단계를 알려주세요."
})
print(f"Q: 그 교수법의 단계를 알려주세요.")
print(f"A: {result3}")

Q: 그 교수법의 단계를 알려주세요.
A: 5단 분석법의 단계는 다음과 같습니다.

1.  기술 용어를 일반 명사로 설명
2.  기술 용어를 고유 명사로 설명
3.  사용 이유 설명
4.  사용 방법 설명
5.  다른 기술과의 비교 정리
